In [1]:
!git clone https://github.com/AlphaTeam-YU/SignLanguageGP.git
%cd SignLanguageGP

Cloning into 'SignLanguageGP'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 102 (delta 38), reused 14 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 35.78 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/SignLanguageGP


In [2]:
%%writefile configs/words.yaml
data:
  raw_videos_dir: "/content/drive/MyDrive/Grad_project/words_videos"
  output_dir: "/content/drive/MyDrive/processed_data/words"
  num_frames: 50
  num_features: 378

extraction:
  model_complexity: 1
  pose_landmarks: 33
  face_landmarks: 40
  hand_landmarks: 21

normalization:
  left_shoulder_idx: 11
  right_shoulder_idx: 12
  eps: 1.0e-6
  use_z_in_scale: false
  min_visibility: 0.5
  frame_width: null
  frame_height: null
  output_x_name: "X_words_norm.npy"
  output_y_name: "y_words.npy"

Overwriting configs/words.yaml


In [3]:
%%writefile words/normalize.py
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import yaml


class WordNormalizer:
    """Shoulder-centered and shoulder-width scale normalizer for holistic word sequences.

    Matches WordLandmarkExtractor output shape: (num_frames, 378) or (N, num_frames, 378)
    Layout per frame (378 features):
      - Pose:       33 landmarks * 4 (x, y, z, visibility) -> [0:132]
      - Face:       40 landmarks * 3 (x, y, z)             -> [132:252]
      - Left hand:  21 landmarks * 3 (x, y, z)             -> [252:315]
      - Right hand: 21 landmarks * 3 (x, y, z)             -> [315:378]
    """

    def __init__(self, config_path: str = "configs/words.yaml") -> None:
        self.config_path = Path(config_path)
        self.config = self._load_config()
        self.norm_cfg = self.config.get("normalization", {})
        self.left_shoulder_idx = int(self.norm_cfg.get("left_shoulder_idx", 11))
        self.right_shoulder_idx = int(self.norm_cfg.get("right_shoulder_idx", 12))
        self.eps = float(self.norm_cfg.get("eps", 1e-6))
        self.use_z_in_scale = bool(self.norm_cfg.get("use_z_in_scale", False))
        self.min_visibility = float(self.norm_cfg.get("min_visibility", 0.0))
        fw = self.norm_cfg.get("frame_width")
        fh = self.norm_cfg.get("frame_height")
        self.default_frame_size: Optional[Tuple[float, float]] = (
            (float(fw), float(fh)) if fw and fh else None
        )

    def _load_config(self) -> dict:
        if not self.config_path.exists():
            return {}
        with self.config_path.open("r", encoding="utf-8") as f:
            return yaml.safe_load(f) or {}

    @staticmethod
    def _missing_mask(block: np.ndarray) -> np.ndarray:
        """(T, K, 3) -> (T, K, 1) bool, True where the point is all-zero (missing)."""
        return np.all(np.isclose(block, 0.0, atol=1e-7), axis=-1, keepdims=True)

    @staticmethod
    def _normalize_xyz_block(
        block: np.ndarray, center: np.ndarray, scale: np.ndarray, eps: float
    ) -> np.ndarray:
        """Centers and scales (T, K, 3) landmarks while preserving missing (all-zero) points."""
        missing_mask = WordNormalizer._missing_mask(block)
        normed = (block - center[:, None, :]) / (scale[:, None, :] + eps)
        return np.where(missing_mask, 0.0, normed)

    @staticmethod
    def _forward_fill(values: np.ndarray, valid: np.ndarray) -> np.ndarray:
        """Forward-fills (T, D) values at invalid frames with the last valid one."""
        if not np.any(valid):
            return np.zeros_like(values)
        T = values.shape[0]
        idx = np.where(valid, np.arange(T), -1)
        idx = np.maximum.accumulate(idx)
        idx[idx < 0] = int(np.argmax(valid))
        return values[idx]

    @staticmethod
    def _apply_frame_size(block: np.ndarray, frame_size: Tuple[float, float]) -> np.ndarray:
        """Converts MediaPipe-normalized coords to a common aspect ratio."""
        w, h = frame_size
        factors = np.array([w, h, w], dtype=np.float32)
        return block * factors

    def normalize_sequence(
        self, seq: np.ndarray, frame_size: Optional[Tuple[float, float]] = None
    ) -> np.ndarray:
        """Normalizes a single video sequence of shape (T, 378)."""
        seq = np.asarray(seq, dtype=np.float32)
        if seq.ndim != 2 or seq.shape[1] != 378:
            raise ValueError(f"Expected sequence shape (T, 378), got {seq.shape}")

        frame_size = frame_size or self.default_frame_size
        T = seq.shape[0]

        pose = seq[:, :132].reshape(T, 33, 4).copy()
        face = seq[:, 132:252].reshape(T, 40, 3).copy()
        left_hand = seq[:, 252:315].reshape(T, 21, 3).copy()
        right_hand = seq[:, 315:378].reshape(T, 21, 3).copy()

        pose_xyz = pose[..., :3]
        pose_vis = pose[..., 3:4]

        if self.min_visibility > 0.0:
            low_vis = pose_vis < self.min_visibility
            pose_xyz = np.where(low_vis, 0.0, pose_xyz)

        if frame_size is not None:
            pose_xyz = self._apply_frame_size(pose_xyz, frame_size)
            face = self._apply_frame_size(face, frame_size)
            left_hand = self._apply_frame_size(left_hand, frame_size)
            right_hand = self._apply_frame_size(right_hand, frame_size)

        l_shoulder = pose_xyz[:, self.left_shoulder_idx, :]
        r_shoulder = pose_xyz[:, self.right_shoulder_idx, :]

        shoulders_present = ~(
            self._missing_mask(l_shoulder[:, None, :])[:, 0, 0]
            | self._missing_mask(r_shoulder[:, None, :])[:, 0, 0]
        )

        center = 0.5 * (l_shoulder + r_shoulder)

        diff = l_shoulder - r_shoulder
        if not self.use_z_in_scale:
            diff = diff[:, :2]
        frame_scale = np.linalg.norm(diff, axis=1, keepdims=True)

        valid_frames = shoulders_present & (frame_scale[:, 0] > self.eps)

        center = self._forward_fill(center, valid_frames)
        if np.any(valid_frames):
            mean_scale = float(np.mean(frame_scale[valid_frames]))
            scale = np.where(valid_frames[:, None], frame_scale, mean_scale)
        else:
            scale = np.ones_like(frame_scale)

        pose_xyz_norm = self._normalize_xyz_block(pose_xyz, center, scale, self.eps)
        face_norm = self._normalize_xyz_block(face, center, scale, self.eps)
        lh_norm = self._normalize_xyz_block(left_hand, center, scale, self.eps)
        rh_norm = self._normalize_xyz_block(right_hand, center, scale, self.eps)

        pose_norm = np.concatenate([pose_xyz_norm, pose_vis], axis=-1)

        return np.concatenate(
            [
                pose_norm.reshape(T, 132),
                face_norm.reshape(T, 120),
                lh_norm.reshape(T, 63),
                rh_norm.reshape(T, 63),
            ],
            axis=1,
        ).astype(np.float32)

    def normalize_batch(
        self, X: np.ndarray, frame_size: Optional[Tuple[float, float]] = None
    ) -> np.ndarray:
        """Normalizes a batch of sequences of shape (N, T, 378)."""
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 3 or X.shape[2] != 378:
            raise ValueError(f"Expected batch shape (N, T, 378), got {X.shape}")
        return np.stack([self.normalize_sequence(seq, frame_size) for seq in X], axis=0)

Writing words/normalize.py


In [4]:
import numpy as np
from words.normalize import WordNormalizer

normalizer = WordNormalizer("configs/words.yaml")
T = 50

# ---- Test 1: all shoulders present
dummy_seq = np.random.uniform(0.1, 0.9, size=(T, 378)).astype(np.float32)
dummy_seq[:, :132].reshape(T, 33, 4)[..., 3] = 1.0  # visibility = 1.0
normed = normalizer.normalize_sequence(dummy_seq)

pose_out = normed[:, :132].reshape(T, 33, 4)
l, r = pose_out[:, 11, :3], pose_out[:, 12, :3]
width = np.linalg.norm((l - r)[:, :2], axis=1)

print("Input shape :", dummy_seq.shape)
print("Output shape:", normed.shape)
print("Max shoulder midpoint residual:", np.max(np.abs(0.5 * (l + r))))
print("Shoulder width after norm (xy) -> min/max:", width.min(), width.max(), "(expect ~1)")

# ---- Test 2: shoulders missing in frames 10..14
broken = dummy_seq.copy()
pose_b = broken[:, :132].reshape(T, 33, 4)
pose_b[10:15, 11, :3] = 0.0
pose_b[10:15, 12, :3] = 0.0
broken[:, :132] = pose_b.reshape(T, 132)

normed_b = normalizer.normalize_sequence(broken)
print("Test 2 finite:", bool(np.all(np.isfinite(normed_b))))
print("Test 2 max |value| (should stay in a sane range):", float(np.max(np.abs(normed_b))))

# ---- Test 3: aspect ratio correction
n_ar = normalizer.normalize_sequence(dummy_seq, frame_size=(1920, 1080))
pose_ar = n_ar[:, :132].reshape(T, 33, 4)
w_ar = np.linalg.norm((pose_ar[:, 11, :2] - pose_ar[:, 12, :2]), axis=1)
print("Test 3 shoulder width with frame_size -> min/max:", w_ar.min(), w_ar.max(), "(expect ~1)")

Input shape : (50, 378)
Output shape: (50, 378)
Max shoulder midpoint residual: 1.937151e-07
Shoulder width after norm (xy) -> min/max: 0.99999344 0.9999988 (expect ~1)
Test 2 finite: True
Test 2 max |value| (should stay in a sane range): 4.663012504577637
Test 3 shoulder width with frame_size -> min/max: 0.9999999 1.0000001 (expect ~1)
